In [1]:
# imports

import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [2]:
load_dotenv(override=True)
ollama_url = "http://localhost:11434/v1"

In [3]:
requests.get("http://localhost:11434/").content

openai = OpenAI()
ollama_url = "http://localhost:11434/v1"
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [4]:
def add_conversation(speaker, message):
    conversations.append(f"{speaker}: {message}")

In [5]:
def judge():
    system_prompt = """
    Role and Identity
    You are an AI adopting the persona of the Chief Justice of the Supreme Court of India. You represent the highest standards of fairness, wisdom, and constitutional knowledge in the country.
    you speak with the calm authority of a senior judge.
        """

    user_prompt = f"""
    Below is the conversation between a prosecutor and a defendant. 
    You are the judge. 
    Please provide the judgement.
    Remember to maintain impartiality and fairness in your assessment.
    keep the judgement concise and to the point, focusing on the key legal and ethical considerations.

    {conversations}
    
    """

    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    return response.choices[0].message.content

In [6]:
def prosecutor():
    system_prompt = """
    You are a prosecutor in a courtroom. Your role is to present the case against the defendant, 
    highlighting the evidence and arguments that support the charges. 
    You should be persuasive, clear, and focused on the facts of the case, 
    while maintaining professionalism and respect for the court and all parties involved.
    You will be allowed to speak {n} times. So make sure to be concise and impactful in your statements 
    and the last statement should be a closing argument.
    While giving the output, do not give the header "Prosecutor:" in the output.
    """
    user_prompt = f"""
    You are a prosecutor in a courtroom. 
    below are the conversations between you and the defendant.

    {conversations}

    Please provide your next statement or question to the defendant. Keep the conversation concise and focused on the case at hand.
    """

    response = ollama.chat.completions.create(
        model="gpt-oss:20b",
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    return response.choices[0].message.content


In [7]:
def defense_lawyer():
    system_prompt = f"""
    You are a defense lawyer in a courtroom. Your role is to vigorously defend your client (the defendant) 
    against the prosecutor's charges, challenge their evidence, and present a compelling counter-narrative.
    You should be persuasive, strategic, and focused on protecting your client's rights, 
    while maintaining professionalism and respect for the court.
    You will be allowed to speak {n} times. So make sure to be concise and impactful in your statements, 
    and your last statement should be a strong closing argument.
    While giving the output, do not give the header "Defense Lawyer:" in the output.
    """
    
    user_prompt = f"""
    You are a defense lawyer in a courtroom. 
    Below is the transcript of the conversations between the legal counsel so far.

    {conversations}

    Please provide your next statement, objection, or response to the prosecutor. Keep the conversation concise and focused on advocating for your client's innocence.
    """

    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    return response.choices[0].message.content

In [8]:
def God():
    system_prompt = """
    You define reality. You know the absolute truth of the case, who the victim is, who the accused is, and what really happened.
    While giving the output, do not give the header "God:" in the output.
    Give the output like below:
    
    <case title>

    Summary: <summary of the case>

    Charges: <charges against the accused>

    Victim: <name of the victim>

    Accused: <name of the accused>

    Evidence: <evidence against the accused>
        """
    user_prompt = """
    Create a case to be resolved in the courtroom. keep it short and concise. The case should be a criminal case, with a clear victim and accused.
    """

    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    return response.choices[0].message.content

In [ ]:
conversations = []  # Conversations between the prosecutor and the defendant
n = 4               # Number of turns each party will speak

case = God()

add_conversation("God", case)

display(Markdown(f"### Case:\n{case}\n"))

for i in range(n):
    prosecutor_statement = prosecutor()
    add_conversation("Prosecutor", prosecutor_statement)
    display(Markdown(f"### Prosecutor:\n{prosecutor_statement}\n"))
    
    defense_response = defense_lawyer()
    add_conversation("Defense", defense_response)
    display(Markdown(f"### Defense:\n{defense_response}\n"))


judgment = judge()

display(Markdown(f"### Judgment:\n{judgment}\n"))


### Case:
Burglary at Elm Street Residence

Summary: On the night of March 12th, a break-in occurred at the Elm Street residence. The homeowner reported missing electronics and jewelry. Security footage captured the suspect entering the home through a window.

Charges: Burglary, breaking and entering, theft

Victim: Sarah Thompson

Accused: Michael Reed

Evidence: Security camera footage showing Michael Reed entering and leaving the residence; stolen items found in Michael Reed’s possession; fingerprints matching Michael Reed on the broken window latch.


### Prosecutor:
Mr. Reed, are you prepared to explain the security footage that places you inside Mrs. Thompson’s home and the matching fingerprints on the broken window latch?


### Defense:
Your Honor, before my client responds, I must object to the admission of the security footage as conclusive evidence. The footage does not establish the context—there is no proof my client unlawfully entered or intended to commit a crime. Furthermore, the provenance and handling of this evidence must be thoroughly scrutinized to rule out any tampering or misidentification. Fingerprints on a window latch do not inherently prove guilt; my client could have come into contact with the property under legitimate circumstances. I ask the court to consider these points carefully before accepting the prosecution's narrative as fact.


### Prosecutor:
Counsel, we have an unbroken, tamper‑evident video recording that shows Mr. Reed at exactly 10:05 p.m.—the time he was on the security system log—entering Mrs. Thompson’s front window and leaving with stolen electronics in his bag. The court will find this footage admissible, as it is directly related to the alleged crime and has an unbroken chain of custody.

The fingerprints collected from the broken latch were analyzed by the same state‑accredited lab that handled the case originally; they match Mr. Reed’s prints at 99.9% confidence with no cross‑matches. The defendant had no legitimate reason or prior permission to be on Mrs. Thompson’s property, nor has he produced any explanation for his presence or possession of the stolen items.

Accordingly, I request that the court enter judgment against Michael Reed as a convicted felon in burglary and theft.

**Closing:** Your Honor, the evidence is clear, incontrovertible, and untainted. Mr. Reed’s continued presence on the property, combined with forensic proof of his fingerprints and possession of stolen goods, leaves no reasonable doubt that he entered Mrs. Thompson’s home unlawfully and committed the theft you now see before you. He should be held accountable for these reprehensible acts.


### Defense:
Your Honor, while the prosecution claims an unbroken chain of custody and conclusive identification, neither the security footage nor the fingerprint evidence addresses critical alternative explanations. My client has a history of visiting Mrs. Thompson’s property legally—as corroborated by multiple witnesses—which explains the presence of his fingerprints and even appearing on security footage previously. The footage shown here captures a brief moment but does not demonstrate intent or illegal entry; it is entirely possible my client entered with permission, unaware of any theft. There remains reasonable doubt as to whether my client committed any crime, given the lack of direct evidence linking him to the act of theft itself. This doubt must be resolved in his favor.
